# Half a million rows, one lookup, a thousand times faster

A database is more than a file that contains a table. It answers questions in a
different way than a loop over rows, it decides where the work happens, and it
guarantees that changes that are only half done never become visible. This
notebook shows these three properties with SQLite, which is built into Python, on
a table of half a million users.

In [1]:
import sqlite3, time
import numpy as np

rng = np.random.default_rng(0)
N = 500_000
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE users (user_id INTEGER, country TEXT, score REAL)")
countries = np.array(["DE", "FR", "IT", "ES", "PL", "NL", "AT", "CH"])
rows = zip(rng.permutation(N).tolist(),                       # user ids in random order
           countries[rng.integers(0, 8, N)].tolist(),
           rng.normal(50, 10, N).round(1).tolist())
db.executemany("INSERT INTO users VALUES (?, ?, ?)", rows)
db.commit()
print(db.execute("SELECT COUNT(*) FROM users").fetchone()[0], "rows")

500000 rows


## 1. Finding one user

We look up 200 users by their id, first with the table as it is and then after
creating an index on `user_id`. `EXPLAIN QUERY PLAN` shows what the database does
in each case.

In [2]:
ids = rng.integers(0, N, 200).tolist()

def lookups():
    t0 = time.perf_counter()
    for i in ids:
        db.execute("SELECT score FROM users WHERE user_id = ?", (i,)).fetchone()
    return time.perf_counter() - t0

print("plan:", db.execute("EXPLAIN QUERY PLAN SELECT score FROM users WHERE user_id = ?", (1,)).fetchone()[-1])
t_scan = lookups()
print(f"200 lookups without index: {t_scan:.2f} s")

db.execute("CREATE INDEX idx_user ON users(user_id)")
print("plan:", db.execute("EXPLAIN QUERY PLAN SELECT score FROM users WHERE user_id = ?", (1,)).fetchone()[-1])
t_index = lookups()
print(f"200 lookups with index:    {t_index:.4f} s   ({t_scan / t_index:.0f}x faster)")

plan: SCAN users


200 lookups without index: 5.69 s


plan: SEARCH users USING INDEX idx_user (user_id=?)
200 lookups with index:    0.0015 s   (3753x faster)


Without the index the plan says `SCAN`, which means that every lookup reads the
whole table of half a million rows to find one row. With the index the plan says
`SEARCH ... USING INDEX`, which means that the database searches a tree of ids
with a few dozen comparisons and goes directly to the row. The lookups become
hundreds to thousands of times faster, and the factor grows with the size of the
table. The index uses the same ideas as the binary search trees and the hash
tables of the Data Structures session, and the database keeps it up to date on
every insert so you do not have to do it yourself. A query that filters on a
column without an index is one of the most common reasons why a database is slow.

## 2. Where the work happens

We want the average score per country. One way is to fetch every row into Python
and aggregate there, and the other way is to ask the database for the answer.
Both are correct, but look at how many rows cross the boundary between the
database and your program.

In [3]:
t0 = time.perf_counter()
all_rows = db.execute("SELECT country, score FROM users").fetchall()
sums, counts = {}, {}
for country, score in all_rows:
    sums[country] = sums.get(country, 0) + score
    counts[country] = counts.get(country, 0) + 1
in_python = {c: round(sums[c] / counts[c], 3) for c in sorted(sums)}
t_py = time.perf_counter() - t0
print(f"fetch everything, aggregate in Python: {t_py:.2f} s, {len(all_rows):,} rows moved")

t0 = time.perf_counter()
in_sql = dict(db.execute("SELECT country, ROUND(AVG(score), 3) FROM users GROUP BY country").fetchall())
t_sql = time.perf_counter() - t0
print(f"GROUP BY in the database:              {t_sql:.2f} s, {len(in_sql)} rows moved")
print("same answer:", in_python == in_sql)

fetch everything, aggregate in Python: 0.94 s, 500,000 rows moved


GROUP BY in the database:              0.40 s, 8 rows moved
same answer: True


The database returns eight rows instead of half a million and it is about twice
as fast even here, where the rows only move within memory. The gap is larger with
a real database on another machine, where every moved row travels over a network.
The rule is to *bring the computation to the data*, i.e. you say what you want in
SQL and let the database decide how to compute it. The mistake to avoid is to
pull a million rows into pandas in order to keep a hundred.

## 3. The transfer that crashes halfway

We have two accounts and a transfer of 30 from Anna to Ben that fails after the
debit and before the credit. This is what happens with a crash, a lost connection
or a bug in the middle of an update. We run it first without a transaction and
then with one.

In [4]:
def transfer(conn, amount, crash):
    conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = 'Anna'", (amount,))
    if crash:
        raise RuntimeError("connection lost")
    conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = 'Ben'", (amount,))

def total(conn):
    return conn.execute("SELECT SUM(balance) FROM accounts").fetchone()[0]

for mode in ["autocommit", "transaction"]:
    conn = sqlite3.connect(":memory:", isolation_level=None if mode == "autocommit" else "DEFERRED")
    conn.execute("CREATE TABLE accounts (name TEXT, balance REAL)")
    conn.execute("INSERT INTO accounts VALUES ('Anna', 100), ('Ben', 100)")
    conn.commit()
    try:
        transfer(conn, 30, crash=True)
        conn.commit()
    except RuntimeError:
        if mode == "transaction":
            conn.rollback()
    print(f"{mode:<12} after the crash: total = {total(conn):.0f}   "
          f"{conn.execute('SELECT name, balance FROM accounts').fetchall()}")

autocommit   after the crash: total = 170   [('Anna', 70.0), ('Ben', 100.0)]
transaction  after the crash: total = 200   [('Anna', 100.0), ('Ben', 100.0)]


In autocommit mode every statement is final on its own. The debit is written and
the credit never happens, so 30 units of money no longer exist anywhere. Inside a
transaction the debit is only provisional until `commit`, and the `rollback` after
the failure restores the state before the transfer began. This is atomicity, the A
of ACID, which means that either the whole transfer happens or none of it does.
The other three properties give similar guarantees against invalid states, with
concurrent users, and after a power cut. A CSV file gives none of these
guarantees.

**What we used:** [what a DBMS does](https://dgarcia-eu.github.io/ICSS/Transcripts/10-Databases.html#what-a-dbms-does),
[transactions and ACID](https://dgarcia-eu.github.io/ICSS/Transcripts/10-Databases.html#transactions-and-acid),
[tables, types, schemas and keys](https://dgarcia-eu.github.io/ICSS/Transcripts/10-Databases.html#tables-types-schemas-and-keys-in-sql)
and [queries with selection, projection and aggregation](https://dgarcia-eu.github.io/ICSS/Transcripts/10-Databases.html#queries-selection-projection-aggregation)
from this session, and [binary search trees](https://dgarcia-eu.github.io/ICSS/Transcripts/03-DataStructures.html#binary-trees-and-binary-search-trees)
from the Data Structures session.